In [1]:
from __future__ import division
import pandas as pd
import numpy as np
from copy import deepcopy

import warnings
warnings.filterwarnings('ignore')

#!pip install py_stringmatching
#!pip install py_entitymatching
#!pip install py_stringsimjoin
import py_stringmatching as sm
import py_entitymatching as em
import py_stringsimjoin as ssj

from py_entitymatching.catalog import catalog_manager as cm

import random

# Introduzione

Questo Notebook riassume tutto il processo di Entity Resolution:
Blocking, Matching e Clustering.


Contiene i punti essenziali per discutere  l' Entity Resolution



# Funzioni utilizzate

In [2]:
######################CLUSTERING######################

import networkx as nx

def ClusterComponentiConnessi(MatchTable, TuttiInodi):

    MatchTable=deepcopy(MatchTable)
    MatchTable.columns=['A','B']

    Singleton = set(TuttiInodi) - set(MatchTable['A']).union(set(MatchTable['B']))

    # Creazione del grafo a partire dagli elementi della MatchTable
    G = nx.Graph()
    for _, row in MatchTable.iterrows():
        G.add_edge(row['A'], row['B'])
#        G.add_edge(row['A'], row['B'], weight=row['sim'])  # Aggiungi il peso (etichetta) basato su 'sim'

    # Aggiungi gli elementi singleton all'insieme dei nodi
    for element in Singleton:
        G.add_node(element)

    # Calcola i componenti connessi (clusters)
    clusters = list(nx.connected_components(G))

    # Creazione del DataFrame dei cluster
    cluster_data = {'ClusterKey': [], 'ClusterElement': []}
    for i, cluster in enumerate(clusters):
        for element in cluster:
            cluster_data['ClusterKey'].append(i + 1)
            cluster_data['ClusterElement'].append(element)

    cluster_df = pd.DataFrame(cluster_data)
    return cluster_df

In [3]:
def VisualizzaCluster2(Clusters):
### NOTA : LA FUNZIONE PRENDE IL NOME DALLA SORGENTE DAI PRIMI CARATTERI
### DELL'IDENTIFICATIVO DEL RECORD
### ADEGUARE IN BASE ALL'ESEMPIO !!!
    vClusters=deepcopy(Clusters)
    vClusters.columns=['ClusterKey', 'ClusterElement']
    vClusters['source']=vClusters['ClusterElement'].astype(str).str[0]

    def Aggregazione(x):
      Campi = {
          '#Sorgenti' :     x['source'].nunique(),
          'Sorgenti' :     x['source'].str.cat(sep=','),
          '#Record' :     x['ClusterElement'].nunique(),
          'Record' :     x['ClusterElement'].str.cat(sep=',')
          }
      return pd.Series(Campi)
    groupedCLUSTERS=vClusters.groupby('ClusterKey').apply(Aggregazione).reset_index()

    return groupedCLUSTERS

def VisualizzaCluster(Clusters,quanti):
### NOTA : LA FUNZIONE PRENDE IL NOME DALLA SORGENTE DAI PRIMI CARATTERI DELL'IDENTIFICATIVO DEL RECORD
### specificere in quanti ; esempio se il record è S2_123, quanti =2 per ottenere S2
    vClusters=deepcopy(Clusters)
    vClusters.columns=['ClusterKey', 'ClusterElement']
    vClusters['source']=vClusters['ClusterElement'].astype(str).str[:quanti]

    def Aggregazione(x):
      Campi = {
          '#Sorgenti' :     x['source'].nunique(),
          'Sorgenti' :     x['source'].str.cat(sep=','),
          '#Record' :     x['ClusterElement'].nunique(),
          'Record' :     x['ClusterElement'].str.cat(sep=',')
          }
      return pd.Series(Campi)
    groupedCLUSTERS=vClusters.groupby('ClusterKey').apply(Aggregazione).reset_index()

    return groupedCLUSTERS


def _VisualizzaDistribuzioneCluster(Clusters):
    gruppi = Clusters.groupby('ClusterKey')
    conteggio_gruppi = gruppi.size().reset_index(name='NumeroElementiPerCluster')
    Risultato=conteggio_gruppi.groupby('NumeroElementiPerCluster').size().reset_index(name='NumeroCluster')
    print("Numero Elementi", (Risultato['NumeroElementiPerCluster'] * Risultato['NumeroCluster']).sum())
    ClusterMax=conteggio_gruppi[conteggio_gruppi['NumeroElementiPerCluster']==Risultato['NumeroElementiPerCluster'].max()]
    print("Cluster con max numero di elementi:", ClusterMax['ClusterKey'].tolist())

    return Risultato

In [4]:
def CalcolaMatchIndottiCluster(Cluster):
  Join=pd.merge(Cluster,Cluster, on='ClusterKey')
  Join=Join[Join.ClusterElement_x<Join.ClusterElement_y]
  Join=Join[['ClusterElement_x','ClusterElement_y']]
  Join.columns=['l_id','r_id']

  return Join.drop_duplicates()

In [5]:
def stable_marriage(MatchTable:pd.DataFrame):
    MATCH = pd.DataFrame(columns=['l_id', 'r_id', "sim"])
    MT = deepcopy(MatchTable)
    MT = MT.sort_values(["sim"], ascending=[False])
    while True:
        R = MT.loc[(~MT['l_id'].isin(MATCH['l_id'])) & (~MT['r_id'].isin(MATCH['r_id']))]
        if len(R) == 0:
            break
        x = R.iloc[0,:]
        MATCH = MATCH.append(x, ignore_index=True)
    return MATCH

def simmetric_best_match(MatchTable:pd.DataFrame):
  CMT = deepcopy(MatchTable)

  CMT['A_RowNo'] = CMT.sort_values(['sim'], ascending=[False]) \
             .groupby(['l_id']) \
             .cumcount() + 1

  CMT['B_RowNo'] = CMT.sort_values(['sim'], ascending=[False]) \
             .groupby(['r_id']) \
             .cumcount() + 1

  return CMT[(CMT.A_RowNo==1) & (CMT.B_RowNo==1)].drop(columns=['A_RowNo', 'B_RowNo']).sort_values(['sim'], ascending=[False])

In [6]:
def Valuta2(Gold:pd.DataFrame, Match:pd.DataFrame):
    Gold = Gold[['l_id','r_id']]
    Match = Match[['l_id','r_id']]
    FOJ = Gold.merge(Match, how='outer', indicator=True)

    TP = FOJ[FOJ['_merge']=='both']
    FP = FOJ[FOJ['_merge']=='right_only']
    FN = FOJ[FOJ['_merge']=='left_only']

    if len(TP) == 0:
        return pd.DataFrame({
                'MT':[len(Match)],
                'TP':[len(TP)],
                'FP':[len(FP)],
                'FN':[len(FN)],
                'P':[round(0,4)],
                'R':[round(0,4)],
                'F':[round(0,4)]
            })
    else:
        P = len(TP)/(len(TP)+len(FP))
        R = len(TP)/(len(TP)+len(FN))
        F = 2 * P * R / ( P + R )
        return pd.DataFrame({
                'MT':[len(Match)],
                'TP':[len(TP)],
                'FP':[len(FP)],
                'FN':[len(FN)],
                'P':[round(P,4)],
                'R':[round(R,4)],
                'F':[round(F,4)]
            })

def VediValuta2(Gold:pd.DataFrame, Match:pd.DataFrame, metrics:str):
    Gold = Gold[['l_id','r_id']]
    Match = Match[['l_id','r_id']]

    FOJ=pd.merge(Gold, Match, how='outer', indicator=True)

    TP=FOJ[FOJ['_merge']=='both']
    FP=FOJ[FOJ['_merge']=='right_only']
    FN=FOJ[FOJ['_merge']=='left_only']

    if metrics == 'FP' :
        return FP
    if metrics == 'TP' :
        return TP
    if metrics == 'FN' :
        return FN

In [7]:
def Valuta(Gold:pd.DataFrame, Match:pd.DataFrame):
 #   Gold = Gold[['l_id','r_id']]
 #   Match = Match[['l_id','r_id']]
    Match = Match.iloc[:, [1, 2]].copy()
    Gold = Gold.iloc[:, [0, 1]].copy()
    Gold.columns=Match.columns=['l_id','r_id']
    FOJ = Gold.merge(Match, how='outer', indicator=True)

    TP = FOJ[FOJ['_merge']=='both']
    FP = FOJ[FOJ['_merge']=='right_only']
    FN = FOJ[FOJ['_merge']=='left_only']

    if len(TP) == 0:
        return pd.DataFrame({
                'MT':[len(Match)],
                'TP':[len(TP)],
                'FP':[len(FP)],
                'FN':[len(FN)],
                'P':[round(0,4)],
                'R':[round(0,4)],
                'F':[round(0,4)]
            })
    else:
        P = len(TP)/(len(TP)+len(FP))
        R = len(TP)/(len(TP)+len(FN))
        F = 2 * P * R / ( P + R )
        return pd.DataFrame({
                'MT':[len(Match)],
                'TP':[len(TP)],
                'FP':[len(FP)],
                'FN':[len(FN)],
                'P':[round(P,4)],
                'R':[round(R,4)],
                'F':[round(F,4)]
            })

def VediValuta(Gold:pd.DataFrame, Match:pd.DataFrame, metrics:str):
    #Gold = Gold[['l_id','r_id']]
    #Match = Match[['l_id','r_id']]
    Match = Match.iloc[:, [1, 2]].copy()
    Gold = Gold.iloc[:, [0, 1]].copy()
    Gold.columns=Match.columns=['l_id','r_id']

    FOJ=pd.merge(Gold, Match, how='outer', indicator=True)

    TP=FOJ[FOJ['_merge']=='both']
    FP=FOJ[FOJ['_merge']=='right_only']
    FN=FOJ[FOJ['_merge']=='left_only']

    if metrics == 'FP' :
        return FP
    if metrics == 'TP' :
        return TP
    if metrics == 'FN' :
        return FN

In [8]:
def ValutaBlocking(DA,DB,Block,Gold):
# INPUT : entrambi Block (è il candidate set after blocking)
#        e Gold (Gold Standard) devono essere con due colonne, l_id e r_id
# per avere indipendenza dal nome di queste colonne
# Si suppone che in Block l_id e r_id siano rispettivamente la seconda e la terza colonna
# e che in Gold sia la prima e la seconda
  Block = Block.iloc[:, [1, 2]].copy()
  Gold = Gold.iloc[:, [0, 1]].copy()
  Gold.columns=Block.columns=['l_id','r_id']

  JOIN=pd.merge(Gold, Block)

 # Reduction_Ratio
  RR=1-len(Block)/(len(DA)*len(DB))
 # Pairs Completeness o Recall
  PC = len(JOIN)/len(Gold)
 # Pairs Quality
  PQ = len(JOIN)/len(Block)

  Risultato = pd.DataFrame([(DA.shape[0],DB.shape[0],Block.shape[0],round(RR,4),round(PC,4),round(PQ,4))],
                             columns=['A', 'B', 'BlockSize', 'ReductRatio','PCompletness','PQuality'])

  return Risultato

In [9]:
def IdSOURCES(Sources:list):
  ListaID= []
  for s in Sources.keys():
    ListaID += Sources[s]['id'].to_list()
  return ListaID

# Entity Resolution tra n sorgenti

## 1) ER  tra due sorgenti : BlockingMatchingRule

Si delinea il processo tra due sorgenti nella funzione **BlockingMatchingRule**

In [10]:
def BlockingMatchingRule(A,B):
    A=deepcopy(A)
    B=deepcopy(B)
    A=A.rename(columns={'id': "l_id" })
    B=B.rename(columns={'id': "r_id" })

    em.set_key(A, 'l_id')
    em.set_key(B, 'r_id')

# BLOCKING

    #...
############# QUI ABBIAMO OTTENUTO IL CANDIDATE SET

# MATCHING


    return MT

## 2) ER tra tutte le coppie di sorgenti: MatchTableSOURCES

La funzione BlockingMatchingRule viene applicata a tutte le coppie di sorgenti : **MatchTableSOURCES**

`MTSOURCES = MatchTableSOURCES(SOURCES)`

In [11]:
def MatchTableSOURCES(Sources:list):
    MatchTable = pd.DataFrame(columns=['l_id', 'r_id', 'sim'])

    for x in Sources.keys():
      for y in Sources.keys():
        if (x<y): # x<=y nel caso dirty

          MTxy = BlockingMatchingRule(Sources[x], Sources[y])

         # global mapping ==> si potrebbe spostare in BlockingMatchingRule
          MTxy = stable_marriage(MTxy.query("l_id!=r_id"))
         # MTxy = simmetric_best_match(MTxy.query("l_id!=r_id"))

          MatchTable=MatchTable.append(MTxy[['l_id', 'r_id', 'sim']], sort=True)
    return MatchTable

## 3) Clusterizzazione: ClusterComponentiConnessi

Si effettua la clusterizzazione dei record sulla base di MTSOURCES usando i componenti connessi
```python
ClusterCalcolati = ClusterComponentiConnessi(
    MTSOURCES[['l_id', 'r_id']],
    IdSOURCES(SOURCES)
)
```
si visualizzano i cluster calcolati

```python
VisualizzaCluster(ClusterCalcolati)
```

e la relativa distribuzione

```python
_VisualizzaDistribuzioneCluster(ClusterCalcolati)
```


## 3) Valutazione tramite **match indotti**

Si confrontano i match indotti dalla clusterizzazione tramite Gold Standard *CalcolaMatchIndottiCluster(ClusterGoldStandard)*
con quelli indotti dalla clusterizzazione calcolata 
*CalcolaMatchIndottiCluster(ClusterCalcolati)*

```python
Valuta(CalcolaMatchIndottiCluster(ClusterGoldStandard),
       CalcolaMatchIndottiCluster(ClusterCalcolati))

```

In [12]:
# Falsi negativi
#VV=VediValuta2(CalcolaMatchIndottiCluster(ClusterGoldStandardCLEAN),CalcolaMatchIndottiCluster(ClusterCalcolati),'FN')
# pd.merge(pd.merge(VV,UNIONE, left_on='l_id', right_on='id'),UNIONE, left_on='r_id', right_on='id')

# Falsi Positivi
## VV=VediValuta2(CalcolaMatchIndottiCluster(ClusterGoldStandardCLEAN),CalcolaMatchIndottiCluster(ClusterCalcolati),'FP')
## pd.merge(pd.merge(VV,UNIONE, left_on='l_id', right_on='id'),UNIONE, left_on='r_id', right_on='id')



## Pre-elaborazione : Unione di tutte le sorgenti

Per semplificare il processo, vengono fornite alcune elaborazioni aggiuntive.

Con il seguente codice si effettua l'UNIONE di tutte le sorgneti in SOURCES
si fissano e si verificano le features da usare nel matching 



 Consideriamo la loro unione nel dataframe UNIONE
 il cui schema sarà quello di una sorgente (hanno tutti lo stesso schema)

```python
UNIONE=pd.DataFrame(columns=SOURCES['S1'].columns)
```



quindi effettuo unione tramite append

```python
for x in SOURCES.keys():
          UNIONE=UNIONE.append(SOURCES[x])
A=UNIONE
B=UNIONE
A=A.rename(columns={'id': "l_id" })
B=B.rename(columns={'id': "r_id" })
em.set_key(A, 'l_id')
em.set_key(B, 'r_id')
F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)
FeaturesList=F['feature_name'].to_list()
print(FeaturesList)
```


In [13]:
# per analizzare i MissedMatches  quelli che sono nel GoldStandard e non nel Candidate Set C
#  esplicitamente con il join\n",
#MissedMatches = GoldStandard.merge(C, on=['l_id', 'r_id'], how='left', 
#                                   indicator=True).query("_merge == 'left_only'")[['l_id', 'r_id']]

#pd.merge(pd.merge(MissedMatches,A),B, on='r_id')


# Esercizio

In [15]:
path='http://dbgroup.ing.unimore.it/EBI/Cluster/'



src_links = [
path+'A.csv',
path+'B.csv',
path+'C.csv']

SOURCES = { 'S'+str(i+1) : pd.read_csv(link).astype(str) for i, link in enumerate(src_links) }

GoldStandard=pd.read_csv(path+ 'ClusterGoldStandard.csv')
GoldStandard


,ClusterKey,ClusterElement
0,1,A_348
1,1,B_2
2,2,B_8
3,2,A_154
4,3,B_10
...,...,...
1870,1581,B_118
1871,1582,C_99
1872,1583,C_83
1873,1584,B_613


In [ ]:
# per analizzare il GoldStandard dato , si calcolano i cluster corrispondenti
ClusterGoldStandard=ClusterComponentiConnessi(GoldStandard[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )
_VisualizzaDistribuzioneCluster(ClusterGoldStandard)

In [15]:
# è un gold standard che clusterizza tutti gli elementi in cluster da due!!

In [16]:
UNIONE=pd.DataFrame(columns=SOURCES['S1'].columns)

for x in SOURCES.keys():
          UNIONE=UNIONE.append(SOURCES[x])
A=UNIONE
B=UNIONE
A=A.rename(columns={'id': "l_id" })
B=B.rename(columns={'id': "r_id" })
em.set_key(A, 'l_id')
em.set_key(B, 'r_id')
F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)
FeaturesList=F['feature_name'].to_list()
print(FeaturesList)

['Nome_Nome_jac_qgm_3_qgm_3', 'Nome_Nome_cos_dlm_dc0_dlm_dc0', 'Nome_Nome_jac_dlm_dc0_dlm_dc0', 'Nome_Nome_mel', 'Nome_Nome_lev_dist', 'Nome_Nome_lev_sim', 'Nome_Nome_nmw', 'Nome_Nome_sw', 'Cognome_Cognome_jac_qgm_3_qgm_3', 'Cognome_Cognome_cos_dlm_dc0_dlm_dc0', 'Cognome_Cognome_jac_dlm_dc0_dlm_dc0', 'Cognome_Cognome_mel', 'Cognome_Cognome_lev_dist', 'Cognome_Cognome_lev_sim', 'Cognome_Cognome_nmw', 'Cognome_Cognome_sw', 'DataNascita_DataNascita_lev_dist', 'DataNascita_DataNascita_lev_sim', 'DataNascita_DataNascita_jar', 'DataNascita_DataNascita_jwn', 'DataNascita_DataNascita_exm', 'DataNascita_DataNascita_jac_qgm_3_qgm_3', 'Sesso_Sesso_lev_dist', 'Sesso_Sesso_lev_sim', 'Sesso_Sesso_jar', 'Sesso_Sesso_jwn', 'Sesso_Sesso_exm', 'Sesso_Sesso_jac_qgm_3_qgm_3', 'Nazionalita_Nazionalita_jac_qgm_3_qgm_3', 'Nazionalita_Nazionalita_cos_dlm_dc0_dlm_dc0', 'Nazionalita_Nazionalita_jac_dlm_dc0_dlm_dc0', 'Nazionalita_Nazionalita_mel', 'Nazionalita_Nazionalita_lev_dist', 'Nazionalita_Nazionalita_lev_

In [17]:
# unione può essere utile anche per visualizzare coppie di record;
# consideriamo ad esempio 5 coppie

Coppie=GoldStandard.head()
print(Coppie)

UNIONE_L = UNIONE.copy().rename(columns={col: f"l_{col}" for col in UNIONE.columns})
UNIONE_R = UNIONE.copy().rename(columns={col: f"r_{col}" for col in UNIONE.columns})
pd.merge(pd.merge(Coppie,UNIONE_L, on='l_id'),UNIONE_R, on='r_id')


   ClusterKey ClusterElement
0           1          A_348
1           1            B_2
2           2            B_8
3           2          A_154
4           3           B_10


KeyError: 'l_id'

In [21]:
# Metodo di Entity Resolution dato
def BlockingMatchingRule(A,B):
    A=deepcopy(A)
    B=deepcopy(B)
    A=A.rename(columns={'id': "l_id" })
    B=B.rename(columns={'id': "r_id" })

    em.set_key(A, 'l_id')
    em.set_key(B, 'r_id')

# BLOCKING
    AttributiOut = [ 'Cognome',  'Sesso', 'Nazionalita',       'CodiceBelfiore']
    

    CandidateDato  = ssj.jaccard_join(A, B, 'l_id', 'r_id',
                                        'CodiceBelfiore', 'CodiceBelfiore',  sm.QgramTokenizer(qval=3), threshold=1,
                                        l_out_attrs=AttributiOut,
                                        r_out_attrs=AttributiOut
                                     )
    CandidateDato=CandidateDato.rename(columns={'l_l_id': 'l_id'})
    CandidateDato=CandidateDato.rename(columns={'r_r_id': 'r_id'})
    CandidateDato=CandidateDato.rename(columns={'_sim_score': 'sim'})
    cm.set_candset_properties(CandidateDato, '_id', 'l_id', 'r_id', A, B)


############# QUI ABBIAMO OTTENUTO IL CANDIDATE SET

# MATCHING
    F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)

    brm = em.BooleanRuleMatcher()
    brm.add_rule(['Cognome_Cognome_jac_qgm_3_qgm_3(ltuple, rtuple)*0.5 \
                   + Nome_Nome_lev_sim(ltuple, rtuple)*0.5 > 0.5' ], F)
    predictions = brm.predict(CandidateDato, target_attr='pred_label', append=True)
    MT=predictions[predictions.pred_label==1]

    return MT

def MatchTableSOURCES(Sources:list):
    MatchTable = pd.DataFrame(columns=['l_id', 'r_id', 'sim'])

    for x in Sources.keys():
      for y in Sources.keys():
        if (x<y): # x<=y nel caso dirty

          MTxy = BlockingMatchingRule(Sources[x], Sources[y])

         # global mapping
          #MTxy = stable_marriage(MTxy.query("l_id!=r_id"))
          MTxy = simmetric_best_match(MTxy)

          MatchTable=MatchTable.append(MTxy[['l_id', 'r_id', 'sim']], sort=True)
    return MatchTable
	
	
MTSOURCES=MatchTableSOURCES(SOURCES)
ClusterCalcolati=ClusterComponentiConnessi(MTSOURCES[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )
_VisualizzaDistribuzioneCluster(ClusterCalcolati)


0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00
0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00
0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


Numero Elementi 1875
Cluster con max numero di elementi: [7, 8, 11, 14, 15, 24, 25, 33, 43, 46, 47, 52, 56, 60, 67, 85, 92, 96, 100, 103, 114, 122, 124, 131, 133, 206]


,NumeroElementiPerCluster,NumeroCluster
0,1,1123
1,2,337
2,3,26


In [19]:
Valuta2(CalcolaMatchIndottiCluster(ClusterGoldStandard),CalcolaMatchIndottiCluster(ClusterCalcolati))

,MT,TP,FP,FN,P,R,F
0,295,279,16,1,0.9458,0.9964,0.9704


**Domanda:** È possibile azzerare i falsi positivi (FP)?  
**Risposta:** Sì, è possibile azzerare i falsi positivi aumentando la soglia di similarità. 


**Domanda:** E per azzerare i falsi negativi (FN), qual è il modo più semplice?  
**Risposta:** Per ridurre o azzerare i falsi negativi, è possibile abbassare la soglia di similarità, così da includere più possibili match. Al limite, si può togliere il matching e restituire l'insieme candidato!!


**Domanda:** Sono sicuro di azzerare i FN? perché ottengo sempre almeno un falso negativo?  
**Risposta:** Perchè c'è il blocking

**Domanda:** Cosa significa un similarity join come  
`'CodiceBelfiore', 'CodiceBelfiore', sm.QgramTokenizer(qval=3), threshold=0.99`  
**Risposta:** Questo comando esegue un similarity join tra due attributi 'CodiceBelfiore', usando un tokenizzatore Q-gram con q=3 e una soglia di similarità di 0.99. In parica equivale a considerare come Blocking Key 'CodiceBelfiore' !!


**Domanda:** Come posso recuperare i falsi negativi, se hanno CodiceBelfiore diverso ma la stessa nazionalità?  
**Risposta:** mettere in *or*/*unione* un secondo blocking basato su nazionalità

**Domanda:** Come includere anche le coppie con stessa nazionalità?  
**Risposta:** similarity join su concatenazione CodiceBelfiore e Mazionalità; unione di due insiemi candidati

In [20]:
# alzo la soglia nel matching

# Metodo di Entity Resolution dato
def BlockingMatchingRule(A,B):
    A=deepcopy(A)
    B=deepcopy(B)
    A=A.rename(columns={'id': "l_id" })
    B=B.rename(columns={'id': "r_id" })

    em.set_key(A, 'l_id')
    em.set_key(B, 'r_id')

# BLOCKING
    AttributiOut = [ 'Cognome',  'Sesso', 'Nazionalita',       'CodiceBelfiore']
    

    CandidateDato  = ssj.jaccard_join(A, B, 'l_id', 'r_id',
                                        'CodiceBelfiore', 'CodiceBelfiore',  sm.QgramTokenizer(qval=3), threshold=1,
                                        l_out_attrs=AttributiOut,
                                        r_out_attrs=AttributiOut
                                     )
    CandidateDato=CandidateDato.rename(columns={'l_l_id': 'l_id'})
    CandidateDato=CandidateDato.rename(columns={'r_r_id': 'r_id'})
    CandidateDato=CandidateDato.rename(columns={'_sim_score': 'sim'})
    cm.set_candset_properties(CandidateDato, '_id', 'l_id', 'r_id', A, B)


############# QUI ABBIAMO OTTENUTO IL CANDIDATE SET

# MATCHING
    F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)

    brm = em.BooleanRuleMatcher()
    brm.add_rule(['Cognome_Cognome_jac_qgm_3_qgm_3(ltuple, rtuple)*0.5 \
                   + Nome_Nome_lev_sim(ltuple, rtuple)*0.5 > 0.8' ], F)
    predictions = brm.predict(CandidateDato, target_attr='pred_label', append=True)
    MT=predictions[predictions.pred_label==1]

    return MT

def MatchTableSOURCES(Sources:list):
    MatchTable = pd.DataFrame(columns=['l_id', 'r_id', 'sim'])

    for x in Sources.keys():
      for y in Sources.keys():
        if (x<y): # x<=y nel caso dirty

          MTxy = BlockingMatchingRule(Sources[x], Sources[y])

         # global mapping
         # MTxy = stable_marriage(MTxy.query("l_id!=r_id"))
         # MTxy = simmetric_best_match(MTxy)

          MatchTable=MatchTable.append(MTxy[['l_id', 'r_id', 'sim']], sort=True)
    return MatchTable
	
	
MTSOURCES=MatchTableSOURCES(SOURCES)
ClusterCalcolati=ClusterComponentiConnessi(MTSOURCES[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )
_VisualizzaDistribuzioneCluster(ClusterCalcolati)



0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00
0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00
0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


Numero Elementi 560
Cluster con max numero di elementi: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 2

,NumeroElementiPerCluster,NumeroCluster
0,1,68
1,2,246


In [21]:
Valuta2(CalcolaMatchIndottiCluster(ClusterGoldStandard),CalcolaMatchIndottiCluster(ClusterCalcolati))

,MT,TP,FP,FN,P,R,F
0,246,246,0,34,1.0,0.8786,0.9354


In [22]:
# abbasso la soglia

# Metodo di Entity Resolution dato
def BlockingMatchingRule(A,B):
    A=deepcopy(A)
    B=deepcopy(B)
    A=A.rename(columns={'id': "l_id" })
    B=B.rename(columns={'id': "r_id" })

    em.set_key(A, 'l_id')
    em.set_key(B, 'r_id')

# BLOCKING
    AttributiOut = [ 'Cognome',  'Sesso', 'Nazionalita',       'CodiceBelfiore']
    

    CandidateDato  = ssj.jaccard_join(A, B, 'l_id', 'r_id',
                                        'CodiceBelfiore', 'CodiceBelfiore',  sm.QgramTokenizer(qval=3), threshold=1,
                                        l_out_attrs=AttributiOut,
                                        r_out_attrs=AttributiOut
                                     )
    CandidateDato=CandidateDato.rename(columns={'l_l_id': 'l_id'})
    CandidateDato=CandidateDato.rename(columns={'r_r_id': 'r_id'})
    CandidateDato=CandidateDato.rename(columns={'_sim_score': 'sim'})
    cm.set_candset_properties(CandidateDato, '_id', 'l_id', 'r_id', A, B)


############# QUI ABBIAMO OTTENUTO IL CANDIDATE SET

# MATCHING
    F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)

    brm = em.BooleanRuleMatcher()
    brm.add_rule(['Cognome_Cognome_jac_qgm_3_qgm_3(ltuple, rtuple)*0.5 \
                   + Nome_Nome_lev_sim(ltuple, rtuple)*0.5 > 0.3' ], F)
    predictions = brm.predict(CandidateDato, target_attr='pred_label', append=True)
    MT=predictions[predictions.pred_label==1]

    return MT

def MatchTableSOURCES(Sources:list):
    MatchTable = pd.DataFrame(columns=['l_id', 'r_id', 'sim'])

    for x in Sources.keys():
      for y in Sources.keys():
        if (x<y): # x<=y nel caso dirty

          MTxy = BlockingMatchingRule(Sources[x], Sources[y])

         # global mapping
         # MTxy = stable_marriage(MTxy.query("l_id!=r_id"))
         # MTxy = simmetric_best_match(MTxy)

          MatchTable=MatchTable.append(MTxy[['l_id', 'r_id', 'sim']], sort=True)
    return MatchTable
	
	
MTSOURCES=MatchTableSOURCES(SOURCES)
ClusterCalcolati=ClusterComponentiConnessi(MTSOURCES[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )
_VisualizzaDistribuzioneCluster(ClusterCalcolati)



0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00
0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00
0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


Numero Elementi 560
Cluster con max numero di elementi: [48]


,NumeroElementiPerCluster,NumeroCluster
0,1,2
1,2,248
2,4,12
3,6,1
4,8,1


In [23]:
Valuta2(CalcolaMatchIndottiCluster(ClusterGoldStandard),CalcolaMatchIndottiCluster(ClusterCalcolati))

,MT,TP,FP,FN,P,R,F
0,363,279,84,1,0.7686,0.9964,0.8678


In [24]:
VV=VediValuta2(CalcolaMatchIndottiCluster(ClusterGoldStandard),CalcolaMatchIndottiCluster(ClusterCalcolati),'FN')
VV

UNIONE_L = UNIONE.copy().rename(columns={col: f"l_{col}" for col in UNIONE.columns})
UNIONE_R = UNIONE.copy().rename(columns={col: f"r_{col}" for col in UNIONE.columns})
pd.merge(pd.merge(VV,UNIONE_L, on='l_id'),UNIONE_R, on='r_id').head()


,l_id,r_id,_merge,l_Nome,l_Cognome,l_DataNascita,l_Sesso,l_Nazionalita,l_CodiceBelfiore,r_Nome,r_Cognome,r_DataNascita,r_Sesso,r_Nazionalita,r_CodiceBelfiore
0,A_194,B_428,left_only,Marco,Giovannetti,20/09/1965,M,Italia,F262,Marco,Giovannetti,10/09/1974,M,Italia,H501


In [25]:
# 
    # BLOCKING by Similarity Join
#    AttributiMix = [ 'Nazionalita',       'CodiceBelfiore']

def BlockingMatchingRule(A,B):
    A=deepcopy(A)
    B=deepcopy(B)
    A=A.rename(columns={'id': "l_id" })
    B=B.rename(columns={'id': "r_id" })

    em.set_key(A, 'l_id')
    em.set_key(B, 'r_id')

# BLOCKING
    AttributiOut = [ 'Cognome',  'Sesso', 'Nazionalita',       'CodiceBelfiore']
    

    # BLOCKING by Similarity Join
    AttributiMix = [ 'Nazionalita',       'CodiceBelfiore']
    
    A['mix'] = A[AttributiMix].astype(str).apply(lambda row: ' '.join(row), axis=1)
    B['mix'] = B[AttributiMix].astype(str).apply(lambda row: ' '.join(row), axis=1)

    
    C  = ssj.jaccard_join(A, B, 'l_id', 'r_id', 
                                        'mix', 'mix',  sm.QgramTokenizer(qval=3), threshold=0.3,
                                        l_out_attrs=AttributiOut,
                                        r_out_attrs=AttributiOut
                                     )
    C=C.rename(columns={'l_l_id': 'l_id'})
    C=C.rename(columns={'r_r_id': 'r_id'})
    C=C.rename(columns={'_sim_score': 'sim'})
    cm.set_candset_properties(C, '_id', 'l_id', 'r_id', A, B)


############# QUI ABBIAMO OTTENUTO IL CANDIDATE SET

# MATCHING
    F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)

    brm = em.BooleanRuleMatcher()
    brm.add_rule(['Cognome_Cognome_jac_qgm_3_qgm_3(ltuple, rtuple)*0.5 \
                   + Nome_Nome_lev_sim(ltuple, rtuple)*0.5 > 0.65' ], F)
    predictions = brm.predict(C, target_attr='pred_label', append=True)
    MT=predictions[predictions.pred_label==1]

    return MT

def MatchTableSOURCES(Sources:list):
    MatchTable = pd.DataFrame(columns=['l_id', 'r_id', 'sim'])

    for x in Sources.keys():
      for y in Sources.keys():
        if (x<y): # x<=y nel caso dirty

          MTxy = BlockingMatchingRule(Sources[x], Sources[y])

         # global mapping
         # MTxy = stable_marriage(MTxy.query("l_id!=r_id"))
         # MTxy = simmetric_best_match(MTxy)

          MatchTable=MatchTable.append(MTxy[['l_id', 'r_id', 'sim']], sort=True)
    return MatchTable

MTSOURCES=MatchTableSOURCES(SOURCES)
ClusterCalcolati=ClusterComponentiConnessi(MTSOURCES[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )
_VisualizzaDistribuzioneCluster(ClusterCalcolati)


0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00
0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00
0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


Numero Elementi 560
Cluster con max numero di elementi: [47]


,NumeroElementiPerCluster,NumeroCluster
0,1,1
1,2,278
2,3,1


In [26]:
Valuta2(CalcolaMatchIndottiCluster(ClusterGoldStandard),CalcolaMatchIndottiCluster(ClusterCalcolati))

,MT,TP,FP,FN,P,R,F
0,281,279,2,1,0.9929,0.9964,0.9947


In [27]:
VV=VediValuta2(CalcolaMatchIndottiCluster(ClusterGoldStandard),CalcolaMatchIndottiCluster(ClusterCalcolati),'FN')
VV

UNIONE_L = UNIONE.copy().rename(columns={col: f"l_{col}" for col in UNIONE.columns})
UNIONE_R = UNIONE.copy().rename(columns={col: f"r_{col}" for col in UNIONE.columns})
pd.merge(pd.merge(VV,UNIONE_L, on='l_id'),UNIONE_R, on='r_id')


,l_id,r_id,_merge,l_Nome,l_Cognome,l_DataNascita,l_Sesso,l_Nazionalita,l_CodiceBelfiore,r_Nome,r_Cognome,r_DataNascita,r_Sesso,r_Nazionalita,r_CodiceBelfiore
0,A_118,B_278,left_only,Said,Rahmouni,07/09/1987,M,Marocco,Z330,Yassine,Rahmouni,01/25/1981,M,Marocco,Z330


In [28]:
VV=VediValuta2(CalcolaMatchIndottiCluster(ClusterGoldStandard),CalcolaMatchIndottiCluster(ClusterCalcolati),'FP')
VV

UNIONE_L = UNIONE.copy().rename(columns={col: f"l_{col}" for col in UNIONE.columns})
UNIONE_R = UNIONE.copy().rename(columns={col: f"r_{col}" for col in UNIONE.columns})
pd.merge(pd.merge(VV,UNIONE_L, on='l_id'),UNIONE_R, on='r_id')


,l_id,r_id,_merge,l_Nome,l_Cognome,l_DataNascita,l_Sesso,l_Nazionalita,l_CodiceBelfiore,r_Nome,r_Cognome,r_DataNascita,r_Sesso,r_Nazionalita,r_CodiceBelfiore
0,B_278,C_203,right_only,Yassine,Rahmouni,01/25/1981,M,Marocco,Z330,Yassine,Al-Rahmouni,14-04-1982,M,Marocco,Z330
1,A_261,B_278,right_only,Yassine,Al-Rahmouni,14/04/1982,M,Marocco,Z330,Yassine,Rahmouni,01/25/1981,M,Marocco,Z330


In [29]:
# 
    # BLOCKING by Similarity Join
#    AttributiMix = [ 'Nazionalita',       'CodiceBelfiore']

def BlockingMatchingRule(A,B):
    A=deepcopy(A)
    B=deepcopy(B)
    A=A.rename(columns={'id': "l_id" })
    B=B.rename(columns={'id': "r_id" })

    em.set_key(A, 'l_id')
    em.set_key(B, 'r_id')

# BLOCKING
    AttributiOut = [ 'Cognome',  'Sesso', 'Nazionalita',       'CodiceBelfiore']
    

    # BLOCKING by Similarity Join
    AttributiMix = [ 'Nazionalita',       'CodiceBelfiore']
    
    A['mix'] = A[AttributiMix].astype(str).apply(lambda row: ' '.join(row), axis=1)
    B['mix'] = B[AttributiMix].astype(str).apply(lambda row: ' '.join(row), axis=1)

    
    C  = ssj.jaccard_join(A, B, 'l_id', 'r_id', 
                                        'mix', 'mix',  sm.QgramTokenizer(qval=3), threshold=0.3,
                                        l_out_attrs=AttributiOut,
                                        r_out_attrs=AttributiOut
                                     )
    C=C.rename(columns={'l_l_id': 'l_id'})
    C=C.rename(columns={'r_r_id': 'r_id'})
    C=C.rename(columns={'_sim_score': 'sim'})
    cm.set_candset_properties(C, '_id', 'l_id', 'r_id', A, B)


############# QUI ABBIAMO OTTENUTO IL CANDIDATE SET

# MATCHING
    F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)

    brm = em.BooleanRuleMatcher()
    brm.add_rule(['Cognome_Cognome_jac_qgm_3_qgm_3(ltuple, rtuple)*0.7 \
                   + Nome_Nome_lev_sim(ltuple, rtuple)*0.3 > 0.7' ], F)
    predictions = brm.predict(C, target_attr='pred_label', append=True)
    MT=predictions[predictions.pred_label==1]

    return MT

def MatchTableSOURCES(Sources:list):
    MatchTable = pd.DataFrame(columns=['l_id', 'r_id', 'sim'])

    for x in Sources.keys():
      for y in Sources.keys():
        if (x<y): # x<=y nel caso dirty

          MTxy = BlockingMatchingRule(Sources[x], Sources[y])

         # global mapping
         # MTxy = stable_marriage(MTxy.query("l_id!=r_id"))
         # MTxy = simmetric_best_match(MTxy)

          MatchTable=MatchTable.append(MTxy[['l_id', 'r_id', 'sim']], sort=True)
    return MatchTable

MTSOURCES=MatchTableSOURCES(SOURCES)
ClusterCalcolati=ClusterComponentiConnessi(MTSOURCES[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )
_VisualizzaDistribuzioneCluster(ClusterCalcolati)


0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00
0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00
0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


Numero Elementi 560
Cluster con max numero di elementi: [24, 89, 204]


,NumeroElementiPerCluster,NumeroCluster
0,2,274
1,4,3


In [30]:
Valuta2(CalcolaMatchIndottiCluster(ClusterGoldStandard),CalcolaMatchIndottiCluster(ClusterCalcolati))

,MT,TP,FP,FN,P,R,F
0,292,280,12,0,0.9589,1.0,0.979


**Considerazioni sul blocking**

In [31]:
# consideriamo solo il blocking
def BlockingMatchingRule(A,B):
    A=deepcopy(A)
    B=deepcopy(B)
    A=A.rename(columns={'id': "l_id" })
    B=B.rename(columns={'id': "r_id" })

    em.set_key(A, 'l_id')
    em.set_key(B, 'r_id')

# BLOCKING
    AttributiOut = [ 'Cognome',  'Sesso', 'Nazionalita',       'CodiceBelfiore']
    

    # BLOCKING by Similarity Join
    AttributiMix = [ 'Nazionalita',       'CodiceBelfiore']
    
    A['mix'] = A[AttributiMix].astype(str).apply(lambda row: ' '.join(row), axis=1)
    B['mix'] = B[AttributiMix].astype(str).apply(lambda row: ' '.join(row), axis=1)

    
    C  = ssj.jaccard_join(A, B, 'l_id', 'r_id', 
                                        'CodiceBelfiore', 'CodiceBelfiore',  sm.QgramTokenizer(qval=3), threshold=1,
                                        l_out_attrs=AttributiOut,
                                        r_out_attrs=AttributiOut
                                     )
    C=C.rename(columns={'l_l_id': 'l_id'})
    C=C.rename(columns={'r_r_id': 'r_id'})
    C=C.rename(columns={'_sim_score': 'sim'})
    cm.set_candset_properties(C, '_id', 'l_id', 'r_id', A, B)
    
    return C

In [32]:
# applichiamolo alle sorgenti S1 e S2
DA=SOURCES['S1'].copy()
DB=SOURCES['S2'].copy()
DA=DA.rename(columns={'id': "l_id" })
DB=DB.rename(columns={'id': "r_id" })
em.set_key(DA, 'l_id')
em.set_key(DB, 'r_id')

CS=BlockingMatchingRule(DA,DB)

0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


In [33]:
# consideriamo il gold standard limitato a queste due sorgenti
GoldStandard=pd.read_csv(path+ 'GOLD_STANDARD__.csv')
GoldStandardS1S2 = GoldStandard[
    GoldStandard['l_id'].str.startswith('A') & GoldStandard['r_id'].str.startswith('B')
].reset_index(drop=True)

In [34]:
ValutaBlocking(DA,DB,CS,GoldStandardS1S2)

,A,B,BlockSize,ReductRatio,PCompletness,PQuality
0,189,179,599,0.9823,0.9886,0.1452


In [35]:
# per analizzare i MissedMatches  quelli che sono nel GoldStandard e non nel Candidate Set C
#  esplicitamente con il join\n",
MissedMatches = GoldStandardS1S2.merge(CS, on=['l_id', 'r_id'], how='left', 
                                   indicator=True).query("_merge == 'left_only'")[['l_id', 'r_id']]
pd.merge(pd.merge(MissedMatches,DA),DB, on='r_id')

,l_id,r_id,Nome_x,Cognome_x,DataNascita_x,Sesso_x,Nazionalita_x,CodiceBelfiore_x,Nome_y,Cognome_y,DataNascita_y,Sesso_y,Nazionalita_y,CodiceBelfiore_y
0,A_194,B_428,Marco,Giovannetti,20/09/1965,M,Italia,F262,Marco,Giovannetti,10/09/1974,M,Italia,H501


In [36]:
# per confrontare le differenti valutazioni usiamo il seguente dataframe
ValutazioneBlocking = pd.DataFrame(columns=['A', 'B', 'BlockSize', 'ReductRatio','PCompletness','PQuality'])

In [37]:
# effettuiamo le seguenti prove (con soglia 1 corrisponde a fare l'equivalence blocker, salvo qualche caso molto particolare)
b1='CodiceBelfiore, soglia 1'
b2='Nazionalita, soglia 1'
b3='Nazionalita + CodiceBelfiore, soglia 0.3'
b4='Nazionalita + CodiceBelfiore, soglia 0.6'
b5='Nazionalita + CodiceBelfiore, soglia 0.5'


In [ ]:
# cambiando la 
ValutazioneBlocking = ValutazioneBlocking.append(
      ValutaBlocking(DA,DB,CS,GoldStandardS1S2)
           ).rename(index={0: b1})
ValutazioneBlocking

In [38]:
# effettuiamo le varie prove; si dovrebbe ottenere il seguente risultato per ValutazioneBlocking
ValutazioneBlockingDF = pd.DataFrame([
    {'TipoBlocking': 'CodiceBelfiore, soglia 1', 'A': 189, 'B': 179, 'BlockSize': 599, 'ReductRatio': 0.9823, 'PCompletness': 0.9886, 'PQuality': 0.1452},
    {'TipoBlocking': 'Nazionalita, soglia 1', 'A': 189, 'B': 179, 'BlockSize': 14521, 'ReductRatio': 0.5708, 'PCompletness': 1.0000, 'PQuality': 0.0061},
    {'TipoBlocking': 'Nazionalita + CodiceBelfiore, soglia 0.3', 'A': 189, 'B': 179, 'BlockSize': 14524, 'ReductRatio': 0.5707, 'PCompletness': 1.0000, 'PQuality': 0.0061},
    {'TipoBlocking': 'Nazionalita + CodiceBelfiore, soglia 0.6', 'A': 189, 'B': 179, 'BlockSize': 650, 'ReductRatio': 0.9808, 'PCompletness': 0.9886, 'PQuality': 0.1338},
    {'TipoBlocking': 'Nazionalita + CodiceBelfiore, soglia 0.5', 'A': 189, 'B': 179, 'BlockSize': 932, 'ReductRatio': 0.9725, 'PCompletness': 0.9886, 'PQuality': 0.0933}
])
ValutazioneBlockingDF

,TipoBlocking,A,B,BlockSize,ReductRatio,PCompletness,PQuality
0,"CodiceBelfiore, soglia 1",189,179,599,0.9823,0.9886,0.1452
1,"Nazionalita, soglia 1",189,179,14521,0.5708,1.0000,0.0061
2,"Nazionalita + CodiceBelfiore, soglia 0.3",189,179,14524,0.5707,1.0000,0.0061
3,"Nazionalita + CodiceBelfiore, soglia 0.6",189,179,650,0.9808,0.9886,0.1338
4,"Nazionalita + CodiceBelfiore, soglia 0.5",189,179,932,0.9725,0.9886,0.0933


In [39]:
# altre considerazioni sul blocking?

# conviene, ha senso fare l'unione dei due blocking su CodiceBelfiore e Nazionalità